# Previsão de Temperatura Global com Machine Learning


Neste notebook, usamos dados históricos de temperatura média global para prever valores futuros (2024, 2044 e 2069) usando um modelo de Regressão Linear.


In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
    

## Carregamento dos Dados e Preparação da Série Temporal

In [4]:

df = pd.read_csv("../data/Environment_Temperature_change_E_All_Data_NOFLAG.csv", encoding="ISO-8859-1")
df.columns = df.columns.str.replace("ï»¿", "")
df = df[df["Element"] == "Temperature change"]

df_long = df.melt(id_vars=["Area", "Months", "Element", "Unit"],
                  var_name="Year", value_name="Value")
df_long["Year"] = pd.to_numeric(df_long["Year"].str.extract(r"(\d+)")[0], errors="coerce")
df_long = df_long.dropna(subset=["Year"])
df_long["Year"] = df_long["Year"].astype(int)

df_long["Value"] = pd.to_numeric(df_long["Value"], errors="coerce")
df_long.dropna(subset=["Value"], inplace=True)
    

## Agrupar Dados Globais por Ano

In [5]:

df_global = df_long[df_long["Months"] == "Meteorological year"]
df_global = df_global.groupby("Year")["Value"].mean().reset_index()
df_global.head()
    

,Year,Value
0,1961,0.170922
1,1962,-0.022873
2,1963,-0.028361
3,1964,-0.106037
4,1965,-0.254930


## Treinamento do Modelo de Regressão Linear

In [ ]:

X = df_global[["Year"]]
y = df_global["Value"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

pred_test = model.predict(X_test)
from math import sqrt
rmse = sqrt(mean_squared_error(y_test, pred_test))

print(f"RMSE: {rmse:.4f}")
    

TypeError: got an unexpected keyword argument 'squared'

## Previsões Futuras para 2024, 2044 e 2069

In [ ]:

anos_futuros = pd.DataFrame({"Year": [2024, 2044, 2069]})
previsoes = model.predict(anos_futuros)

for ano, temp in zip(anos_futuros["Year"], previsoes):
    print(f"Previsão para {ano}: {temp:.2f} °C")
    

## Visualização da Tendência e Previsões

In [ ]:

plt.figure(figsize=(10, 6))
sns.scatterplot(x="Year", y="Value", data=df_global, label="Histórico")
sns.lineplot(x=df_global["Year"], y=model.predict(df_global[["Year"]]), label="Regressão Linear", color="green")
plt.scatter(anos_futuros["Year"], previsoes, color="red", marker="X", s=100, label="Previsões")
plt.title("Previsão de Temperatura Global")
plt.xlabel("Ano")
plt.ylabel("Temperatura Média (°C)")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()
    